# Crop-Irrigation-AI — Exploratory Data Analysis

This notebook explores the ground-truth survey data, weather time series, and
command-area zone boundaries used throughout the pipeline.

**Sections**
1. Setup
2. Ground-truth crop label distribution
3. Spatial distribution of survey points
4. Weather time-series exploration
5. Command-area zone summary


In [ ]:
# 1. Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from config import settings

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True


## 2. Ground-truth crop label distribution

In [ ]:
gt = pd.read_csv(settings.ground_truth_csv)
print(f"Total survey points: {len(gt)}")
gt.head()


In [ ]:
fig, ax = plt.subplots()
gt['crop_class'].value_counts().plot(kind='bar', ax=ax, color='#1D9E75')
ax.set_title('Ground-truth crop class distribution')
ax.set_xlabel('Crop')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots()
gt.boxplot(column='ndvi_observed', by='crop_class', ax=ax, rot=45)
ax.set_title('Observed NDVI by crop class')
plt.suptitle('')
plt.tight_layout()
plt.show()


## 3. Spatial distribution of survey points

In [ ]:
gdf = gpd.GeoDataFrame(
    gt, geometry=gpd.points_from_xy(gt.lon, gt.lat), crs='EPSG:4326'
)

boundary = gpd.read_file(settings.boundary_shp)

fig, ax = plt.subplots(figsize=(8, 8))
boundary.plot(ax=ax, facecolor='none', edgecolor='#1D9E75', linewidth=1.5)
gdf.plot(ax=ax, column='crop_class', legend=True, markersize=15,
         legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)})
ax.set_title('Survey points over command-area zones')
plt.tight_layout()
plt.show()


## 4. Weather time-series exploration

In [ ]:
weather_files = list(settings.weather_dir.glob('*.csv'))
print('Weather files found:', weather_files)

weather = pd.read_csv(weather_files[0], parse_dates=['date'])
weather.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(weather['date'], weather['temp_max'], label='Max Temp (°C)', color='#D85A30')
axes[0].plot(weather['date'], weather['temp_min'], label='Min Temp (°C)', color='#378ADD')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[0].set_title('Daily Temperature Range')

axes[1].bar(weather['date'], weather['precip_mm'], color='#1D9E75', width=0.8, label='Rainfall (mm)')
axes[1].plot(weather['date'], weather['et0_mm'], color='#854F0B', label='ET₀ (mm)')
axes[1].set_ylabel('mm')
axes[1].legend()
axes[1].set_title('Rainfall vs Reference Evapotranspiration')

plt.tight_layout()
plt.show()


In [ ]:
print('Weather summary statistics:')
weather[['temp_mean','precip_mm','et0_mm','humidity_pct']].describe()


## 5. Command-area zone summary

In [ ]:
print(f"Total zones: {len(boundary)}")
print(f"Total command area: {boundary['area_ha'].sum():,.1f} ha")
boundary[['zone_id', 'zone_name', 'area_ha']].sort_values('area_ha', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
boundary.plot(column='area_ha', cmap='Greens', edgecolor='black', legend=True, ax=ax)
for _, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(row['zone_id'], (centroid.x, centroid.y), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Command Area Zones by Size')
plt.tight_layout()
plt.show()


## Next steps

Proceed to **`model_training.ipynb`** to train the crop classifier, moisture-stress
CNN, and water-deficit regressor using features derived from this exploratory
analysis.
